# Encontro 2 - Insper AI - Regressão Logística e Gradiente Descendente

---
*Autor: Thomas Kassabian - Diretor da Insper AI*  
*Colaboradores: --*  
*Atualizado em: 09/2025*  

---

O objetivo dessa atividade é implementar e treinar manualmente um classificador binário usando Regressão Logística e Gradiente Descendente.

### Preparação inicial: importando bibliotecas, dados e pré-processamento

Não precisa fazer nada nessa seção. Apenas execute as células.

In [24]:
# Importar as bibliotecas necessárias
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

cancer = load_breast_cancer(as_frame=True)
df = cancer.frame

In [25]:
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Tamanho do conjunto de treino:", X_train.shape, y_train.shape)
print("Quantidade de features:", X_train.shape[1])
print("Tamanho do conjunto de teste:", X_test.shape, y_test.shape)

Tamanho do conjunto de treino: (455, 30) (455,)
Quantidade de features: 30
Tamanho do conjunto de teste: (114, 30) (114,)


---
### A Teoria da Regressão Logística

Diferente da Regressão Linear, que prevê um **valor contínuo**, a Regressão Logística prevê uma **probabilidade**, que é então usada para classificar a entrada em uma de duas categorias. Portanto, a variável target é binária (ex: Sim/Não, Doente/Saudável, 0/1).

#### O Desafio: De Linha a Probabilidade

Se usássemos a equação da Regressão Linear ($\hat{y} = \mathbf{w}^T \cdot \mathbf{x} + b$) para um problema de classificação, teríamos um problema: a saída pode ser qualquer número real (de $-\infty$ a $+\infty$), e não uma probabilidade que deve, por definição, estar entre 0 e 1.

A solução da Regressão Logística é aplicar uma função "esmagadora" ao resultado da equação linear. Essa função é chamada de **Função Sigmoide** (ou Função Logística), que é um exemplo de **função de ativação**.

#### Função de Ativação
Uma função de ativação é usada em modelos de machine learning para **introduzir não-linearidade**, permitindo que o modelo capture relações complexas nos dados. No caso da Regressão Logística, a função de ativação transforma a saída linear em uma probabilidade, facilitando a classificação.

#### Função Sigmoide

A Função de Ativação Sigmoide, denotada por $\sigma(z)$, tem a seguinte fórmula:

$$\sigma(z) = \frac{1}{1 + e^{-z}}$$

Ela pega qualquer valor de entrada $z$ e o transforma em um valor entre 0 e 1, que podemos interpretar como uma probabilidade.

![Função Sigmoide](https://upload.wikimedia.org/wikipedia/commons/thumb/8/88/Logistic-curve.svg/600px-Logistic-curve.svg.png)

#### A Equação Completa

A equação da Regressão Logística combina a equação linear com a função sigmoide:

1.  Primeiro, calcula-se uma pontuação linear (exatamente como na Regressão Linear):
    $$z = \mathbf{w}^T \cdot \mathbf{x} + b$$

2.  Em seguida, essa pontuação é passada pela função sigmoide para obter a probabilidade:
    $$\hat{p} = \sigma(z) = \frac{1}{1 + e^{-(\mathbf{w}^T \cdot \mathbf{x} + b)}}$$

Onde:
* $\hat{p}$: a probabilidade prevista de a amostra pertencer à classe positiva (classe 1).
* $\mathbf{w}$ e $b$: são os vetores de pesos e bias que o modelo precisa aprender.

O "aprendizado" aqui significa encontrar os melhores valores de $\mathbf{w}$ e $b$ que fazem o modelo prever probabilidades altas para as amostras da classe positiva e probabilidades baixas para as da classe negativa.

A previsão final do modelo é feita com base em um limiar (threshold), geralmente 0.5:
- Se $\hat{p} \geq 0.5$, classifica como classe 1 (classe positiva).
- Se $\hat{p} < 0.5$, classifica como classe 0 (classe negativa).

### Implementação

#### Passo 1: Inicialização dos Parâmetros
A primeira etapa é inicializar os pesos $\mathbf{w}$ e o bias $b$. Uma prática comum é iniciar os pesos com valores pequenos aleatórios e o bias com zero.

**Dicas**:
- Crie uma função `initialize_parameters(n_features)` que retorna um vetor de pesos e um bias inicializados.

- Use `numpy` para criar arrays de pesos.
- Inicialize os pesos com valores aleatórios pequenos (ex: `np.random.randn(n_features) * 0.01`).

In [ ]:
# SEU CÓDIGO AQUI

def initialize_parameters(n_features):
    """
    Inicializa os pesos e o bias.
    
    Argumentos:
    n_features -- número de features no conjunto de dados
    
    Retorna:
    w -- vetor de pesos inicializados
    b -- bias inicializado
    """

#### Passo 2: Função Sigmoide
Implemente a função sigmoide que transforma a saída linear em uma probabilidade.

**Dicas**:
- Crie uma função `sigmoid(z)` que retorna a sigmoide de `z`.

- Use `numpy` para calcular a sigmoide de forma vetorizada.
- Use a função `np.exp()` para calcular a exponencial.

- Reflita sobre o uso da função `np.clip()` no código fornecido. Por que é importante limitar os valores de `z`?

In [35]:
def sigmoid(z):
    """
    Calcula a função sigmoide.
    
    Argumentos:
    z -- valor escalar ou array numpy
    
    Retorna:
    s -- resultado da função sigmoide aplicada a z
    """
    z = np.clip(z, -500, 500)

#### Passo 3: Fazendo previsões
Implemente a função que faz as previsões usando os pesos e bias atuais.


**Dicas**:
- Crie uma função `predict(X, w, b)` que retorna uma **lista** com as previsões das probabilidades para um conjunto de dados `X`.  

- Use a função `sigmoid()` para calcular as probabilidades.  
- Use a função np.dot() para calcular o produto escalar entre `X` e `w` (lembre-se que `w` é um vetor de pesos, um para cada feature de `X`). Isso possibilita a implementação da função sem loops (substancialmente mais eficiente e é a prática padrão). 

    Se você não conhece ou lembra, pesquise o que é e como funciona um **produto escalar**. 

In [36]:
def predict(X, w, b):
    """
    Faz previsões usando os pesos e bias atuais.
    
    Argumentos:
    X -- conjunto de dados (numpy array de shape (m, n_features))
    w -- vetor de pesos (numpy array de shape (n_features,))
    b -- bias (float)
    
    Retorna:
    y_pred -- previsões (numpy array de shape (m,))
    """

---

### A Função de Custo

Não podemos usar o Erro Quadrático Médio (MSE) da Regressão Linear, pois quando combinado com a função sigmoide, ele cria uma função de custo não-convexa, cheia de mínimos locais, dificultando a otimização.

Para a Regressão Logística, usamos uma função de custo chamada **Log Loss**, ou **Entropia Cruzada Binária (Binary Cross-Entropy)**. A intuição por trás dela é simples: ela penaliza fortemente o modelo quando ele está "confiantemente errado".

* Se a classe real é 1, a função de custo penaliza o modelo à medida que a probabilidade prevista $\hat{p}$ se aproxima de 0.
* Se a classe real é 0, a função de custo penaliza o modelo à medida que a probabilidade prevista $\hat{p}$ se aproxima de 1.

A fórmula completa que combina os dois casos é:

$$J(\mathbf{w}, b) = - \frac{1}{m} \sum_{i=1}^{m} [y_i \log(\hat{y}_i) + (1 - y_i) \log(1 - \hat{y}_i)]$$

Onde:
- $m$: número de exemplos no conjunto de dados.
- $y_i$: classe real do exemplo $i$ (0 ou 1).
- $\hat{y}_i$: probabilidade prevista pelo modelo para o exemplo $i$.

Assim como antes, a função de custo depende dos parâmetros do modelo ($\mathbf{w}$ e $b$). Treinar o modelo significa encontrar os valores ótimos de $\mathbf{w}$ e $b$ que minimizam essa função de custo, e isso também é feito através de algoritmos de otimização como o **Gradiente Descendente**.

Os vídeos 34 e 35 do curso de Machine Learning devem ajudar a entender a função de custo Log Loss: https://www.youtube.com/watch?v=vq4Ie5xWhww&list=PLkDaE6sCZn6FNC6YRfRQc_FbeQrF8BwGI&index=34

### Implementação
Implemente uma função que calcula a função de custo Log Loss.

**Dicas**:
- Crie uma função `cost_function(X, y, w, b)` que retorna o valor da função de custo.

- Use a função `predict()` para obter as probabilidades previstas.
- Use a função `np.log()` para calcular o logaritmo.
- Você pode implementar a somatória usando um loop ou de forma vetorizada com `np.sum()`.

- IMPORTANTE: Para evitar problemas de log(0), você pode adicionar um pequeno valor (ex: `1e-15`) às probabilidades previstas antes de calcular o logaritmo.

In [37]:
def cost_function(X, y, w, b):
    """
    Calcula a função de custo logístico.
    
    Argumentos:
    X -- conjunto de dados (numpy array de shape (m, n_features))
    y -- rótulos verdadeiros (numpy array de shape (m,))
    w -- vetor de pesos (numpy array de shape (n_features,))
    b -- bias (float)
    
    Retorna:
    cost -- valor da função de custo
    """

---
### Otimização: Gradiente Descendente

#### Introdução
O Gradiente Descendente é um algoritmo de otimização usado para **minimizar a função de custo** ajustando os parâmetros do modelo ($\mathbf{w}$ e $b$). 

A ideia básica é calcular o **gradiente da função de custo** em relação aos parâmetros e, em seguida, atualizar os parâmetros na direção oposta ao gradiente.

#### O gradiente da função de custo
O gradiente é um vetor que aponta na **direção de maior crescimento** de uma função, sendo composto pelas derivadas parciais em relação a cada parâmetro.

**Derivada parcial**: Assim como a derivada representa a **taxa de crescimento** de uma função, a derivada parcial representa a taxa em relação a um parâmetro específico. Ou seja, se fixarmos todos os parâmetros, exceto um, a derivada parcial nos dirá como a função de custo muda quando alteramos apenas aquele parâmetro.

   $$\nabla J(\mathbf{w}, b) = \left[ \frac{\partial J}{\partial \mathbf{w}}, \frac{\partial J}{\partial b} \right]$$

Onde:
- $\nabla$ é o símbolo do gradiente.
- $\frac{\partial }{\partial \mathbf{w}}$ representa a derivada parcial em relação ao parâmetro correspondente (nesse caso $\mathbf{w}$).

Para a Regressão Logística, as derivadas parciais da função de custo Log Loss são:

$$\frac{\partial J}{\partial \mathbf{w_j}} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i) \cdot \mathbf{x_j}_i$$
$$\frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} (\hat{y}_i - y_i)$$

Onde:
- $m$: número de exemplos no conjunto de dados.
- $i$: índice do exemplo atual.
- $j$: índice da feature atual.

Fique à vontade para deduzir essas fórmulas, mas não é obrigatório para a implementação.

O vídeo 36 do curso de Machine Learning deve ajudar a entender as derivadas parciais: https://www.youtube.com/watch?v=H9bXvYh3nO4&list=PLkDaE6sCZn6FNC6YRfRQc_FbeQrF8BwGI&index=36

#### Método do Gradiente Descendente

Como você pode ter imaginado, o método é iterativo, e consiste nos seguintes passos:

1. **Inicialização**: Começamos com valores iniciais para $\mathbf{w}$ e $b$ (geralmente zeros ou pequenos valores aleatórios).

2. **Cálculo do Gradiente**: Para cada iteração, calculamos o gradiente da função de custo:

   $$\nabla J(\mathbf{w}, b) = \left[ \frac{\partial J}{\partial \mathbf{w}}, \frac{\partial J}{\partial b} \right]$$

3. **Atualização dos Parâmetros**: Atualizamos os parâmetros na direção oposta ao gradiente:

   $$\mathbf{w} \leftarrow \mathbf{w} - \alpha \frac{\partial J}{\partial \mathbf{w}}$$
   $$b \leftarrow b - \alpha \frac{\partial J}{\partial b}$$

   Onde $\alpha$ é a taxa de aprendizado, um hiperparâmetro que controla o tamanho dos passos de atualização.

4. **Iteração**: Repetimos os passos 2 e 3 até que a função de custo converja (ou seja, até que as atualizações se tornem muito pequenas) ou até atingir um número máximo de iterações.

Os vídeos 15 a 17 do curso de Machine Learning devem ajudar nessa parte: https://www.youtube.com/watch?v=WtlvKq_zxPI&list=PLkDaE6sCZn6FNC6YRfRQc_FbeQrF8BwGI&index=16

### Implementação

#### Passo 1: Cálculo do Gradiente
Implemente uma função que calcula o gradiente da função de custo em relação aos parâmetros $\mathbf{w}$ e $b$.

**Dicas**:
- Crie uma função `gradient(X, y, w, b)` que retorna as derivadas parciais $\frac{\partial J}{\partial \mathbf{w}}$ e $\frac{\partial J}{\partial b}$.

- Use a função `predict()` para obter as probabilidades previstas. Lembre-se que predict retorna um **array** (lista) de probabilidades.
- Lembre-se que cada elemento de `w`, ou seja, cada peso de uma feature, tem sua própria derivada parcial.
- Crie uma lista ou array para armazenar as derivadas parciais de cada peso.

- Implemente a função de forma vetorizada usando `np.dot()` para eficiência, assim como você fez na função `predict()`.  Se você não fez, volte e implemente. Se você não conhece, procure o que é e como funciona um **produto escalar**.

In [38]:
def gradient(X, y, w, b):
    """
    Calcula o gradiente da função de custo de forma vetorizada.
    
    Argumentos:
    X -- conjunto de dados (numpy array de shape (m, n_features))
    y -- rótulos verdadeiros (numpy array de shape (m,))
    w -- vetor de pesos (numpy array de shape (n_features,))
    b -- bias (float)
    
    Retorna:
    dj_dw -- vetor de derivadas parciais em relação a w
    dj_db -- derivada parcial em relação a b
    """

#### Passo 2: Implementação do Gradiente Descendente
Implemente a função que executa o algoritmo do Gradiente Descendente para otimizar os parâmetros $\mathbf{w}$ e $b$.

**Dicas**:
- Crie uma função `gradient_descent(X, y, w_inicial, b_inicial, alpha, num_iterations)` que retorna os parâmetros otimizados.
- Use a função `gradient()` para calcular as derivadas parciais.
- Atualize os parâmetros $\mathbf{w}$ e $b$ usando as fórmulas de atualização.
- Use a função `cost_function()` para calcular o valor da função de custo a cada iteração e armazene esses valores para análise posterior.
- Adicione prints para mostrar o valor da função de custo a cada 100 iterações (ou outro intervalo que você achar interessante).
- Considere adicionar um critério de parada baseado na convergência da função de custo (ex: se a mudança na função de custo for menor que um certo limiar).

In [ ]:
def gradient_descent(X, y, w_inicial, b_inicial, alpha, num_iterations):
    """
    Executa o algoritmo de Gradiente Descendente.
    
    Argumentos:
    X -- conjunto de dados (numpy array de shape (m, n_features))
    y -- rótulos verdadeiros (numpy array de shape (m,))
    w_inicial -- vetor de pesos inicial (numpy array de shape (n_features,))
    b_inicial -- bias inicial (float)
    alpha -- taxa de aprendizado (float)
    num_iterations -- número de iterações para o gradiente descendente (int)
    
    Retorna:
    w -- vetor de pesos final (numpy array de shape (n_features,))
    b -- bias final (float)
    J_history -- lista com os valores da função de custo em cada 100 iterações
    """

#### Passo 3: Aplicando o Gradiente Descendente
Agora, com todas as funções implementadas, treine o modelo usando o Gradiente Descendente.

**Dicas**:
- Use a função `initialize_parameters()` para obter os pesos e bias iniciais.
- Defina uma taxa de aprendizado (ex: `alpha = 0.001`) e um número máximo de iterações (ex: `num_iterations = 10000`, o mesmo do outro notebook).
- Chame a função `gradient_descent()` com os parâmetros apropriados para treinar o modelo.
- Após o treinamento, imprima os pesos e bias finais.

In [39]:
# SEU CÓDIGO AQUI

#### Compare

Você deve ter exibido os parâmetros do modelo treinado pelo scikit-learn no outro notebook. Compare esses parâmetros com os que você obteve manualmente. Eles devem ser muito semelhantes, embora possam não ser exatamente iguais devido a diferenças na inicialização, taxa de aprendizado, número de iterações e outros fatores.

--- 
### Avaliação do Modelo

Após treinar o modelo, é importante avaliar seu desempenho usando métricas apropriadas para problemas de classificação binária.

**Dicas**:
- Use a função `predict()` para obter as previsões do modelo no conjunto de teste.
- Calcule a acurácia, matriz de confusão, precisão, recall e F1-score. Sinta-se à vontade para usar as funções do `sklearn.metrics` para facilitar esses cálculos.
- Compare as previsões com as classes reais para calcular as métricas.

In [40]:
# SEU CÓDIGO AQUI

### Conclusão

Neste notebook, você implementou um classificador de Regressão Logística a partir do zero, utilizando o algoritmo de otimização de Gradiente Descendente (Batch) para treinar os parâmetros do modelo. Compare o desempenho e os parâmetros do modelo manual com a implementação padrão da biblioteca `scikit-learn` feita no outro notebook.

**Principais Aprendizados:**

* **Abertura da "Caixa-Preta":** A atividade demonstrou na prática o mecanismo interno de um classificador linear e esclareceu o processo de "treinamento":
    1.  Definição de uma **Função de Custo** (Log Loss) para quantificar o erro.
    2.  Cálculo do **Gradiente** para determinar a direção de maior erro.
    3.  Ajuste iterativo dos **Parâmetros** ($\mathbf{w}$ e $b$) na direção oposta ao gradiente para minimizar o custo.

* **Teoria vs. Código:** A relação entre a teoria matemática (cálculo das derivadas parciais) e a implementação prática (código em Python/NumPy) foi estabelecida de forma concreta.

* **Impacto dos Hiperparâmetros:** A importância de hiperparâmetros como a **taxa de aprendizado ($\alpha$)** e o **número de iterações** se tornou tangível, impactando diretamente a convergência e o desempenho final do modelo.

Este conhecimento é fundamental para entender o funcionamento interno de um neurônio artificial e seu processo de otimização (Gradiente Descendente), que é a base fundamental para o próximo encontro, onde construiremos a nossa primeira Rede Neural.